# non-Multilayers process

In [14]:
!pip install pennylane

In [15]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [16]:
class QuanvolutionLayer(nn.Module):
    def __init__(self):
        super(QuanvolutionLayer, self).__init__()
        self.q_weights = nn.Parameter(0.01 * torch.rand((1, n_qubits)))  # Quantum trainable weights
        self.wev_weights = nn.Parameter(torch.randn(n_qubits, 6))  # Trainable weight matrix for WEV
        self.bias = nn.Parameter(torch.randn(6))  # Bias for WEV

    def forward(self, x):
        batch_size, height, width, channels = x.shape
        output = torch.zeros(batch_size, height - 1, width - 1, n_qubits)

        for i in range(height - 1):
            for j in range(width - 1):
                for b in range(batch_size):
                    patch = x[b, i:i+2, j:j+2, :].reshape(-1)  # Extract 2x2 patch
                    q_out = torch.tensor(quantum_circuit(patch, self.q_weights), dtype=torch.float32).detach()  # Quantum features
                    output[b, i, j, :] = q_out

        # Apply WEV transformation (weighted sum over quantum features)
        output = output @ self.wev_weights + self.bias  # Shape: (batch_size, height-1, width-1, 6)

        return output

In [17]:
class QuanvolutionalNN(nn.Module):
    def __init__(self):
        super(QuanvolutionalNN, self).__init__()
        self.quanv = QuanvolutionLayer()
        self.conv = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, padding=1)  # Match WEV outputs
        self.fc1 = nn.Linear(16 * 6 * 6, 64)
        self.fc2 = nn.Linear(64, 6)  # Output 6 values (one per depth)

    def forward(self, x):
      x = self.quanv(x)  # Output shape: (batch_size, height-1, width-1, 6)
      x = x.permute(0, 3, 1, 2)  # Rearrange to (batch, channels, height, width)
      x = torch.relu(self.conv(x))
      x = torch.flatten(x, start_dim=1)
      x = torch.relu(self.fc1(x))
      x = self.fc2(x)  # Now outputs (batch_size, 6)
      return x.unsqueeze(-1)  # Ensure shape (batch_size, 6, 1)



In [18]:
num_samples = 100
data = np.load('inputs.npy')
labels = np.load('outputpH.npy')
print(data.shape, labels.shape)
labels = torch.tensor(labels, dtype=torch.float32)

(177, 7, 7, 6) (177, 6, 1)


In [19]:


model = QuanvolutionalNN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 40
for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(data)
    print(type(outputs), type(labels))
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    print(f"epoch {epoch+1}/{num_epochs}\n loss: {loss.item():.4f}")


<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 1/40
 loss: 41.2195
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 2/40
 loss: 23.0893
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 3/40
 loss: 11.1283
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 4/40
 loss: 13.7538
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 5/40
 loss: 8.1722
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 6/40
 loss: 10.5351
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 7/40
 loss: 11.8494
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 8/40
 loss: 10.5048
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 9/40
 loss: 8.6477
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 10/40
 loss: 8.6877
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 11/40
 loss: 9.0483
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 12/40
 loss: 7.5418
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 13/40
 loss: 6.6850
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 14/40
 loss: 7

In [20]:
torch.save(model.state_dict(), "quanvolutional_model.pth")